# Stock Price Prediction — Multi-Model Comparative Analysis

This notebook walks through the full pipeline end-to-end:

1. Download historical OHLCV data via `yfinance`
2. Engineer 40+ technical indicators
3. Train and evaluate 12+ ML models with a strict temporal split
4. Cross-stock comparison (AAPL, MSFT, JPM, JNJ, XOM)
5. Key findings and visualisations

All reusable logic lives in `src/stock_prediction/`; the notebook contains only orchestration and visualisation calls.

## 0 · Setup

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.style as mstyle
mstyle.use("seaborn-v0_8-darkgrid")
%matplotlib inline

# Project imports
from stock_prediction.config         import STOCKS, START_DATE, END_DATE, PREDICTION_HORIZON
from stock_prediction.data           import download_stocks
from stock_prediction.features       import engineer_features, prepare_xy
from stock_prediction.models         import build_models, train_pipeline, run_all_stocks
from stock_prediction.models.evaluate import build_comparison_table
from stock_prediction.visualization  import (
    plot_predictions, plot_residuals, plot_model_comparison,
    plot_cross_stock_comparison, plot_feature_importance,
)

print(f"Stocks  : {list(STOCKS.keys())}")
print(f"Period  : {START_DATE} → {END_DATE}")
print(f"Horizon : {PREDICTION_HORIZON} trading days")

## 1 · Download Data

In [ ]:
stock_data = download_stocks()

for ticker, df in stock_data.items():
    print(f"{ticker:6s}  {len(df):,} trading days  "
          f"({df.index.min().date()} → {df.index.max().date()})")

## 2 · Feature Engineering (AAPL preview)

The same pipeline is applied to all tickers. Here we inspect the feature set for AAPL.

In [ ]:
df_aapl = engineer_features(stock_data["AAPL"], "AAPL")
X_aapl, y_aapl, feat_cols = prepare_xy(df_aapl, "AAPL")

print(f"Rows    : {len(X_aapl):,}")
print(f"Features: {len(feat_cols)}")
print()

# Top correlations with the target
corr = X_aapl.corrwith(y_aapl).abs().sort_values(ascending=False)
print("Top 10 features by |correlation| with Target:")
print(corr.head(10).to_string())

## 3 · Full Model Comparison (AAPL)

Train all 12+ models on AAPL with a strict **temporal 80/20 split** — no shuffling, no data leakage.

In [ ]:
from stock_prediction.models.train import build_models, build_ensemble

base   = build_models()
ensemb = build_ensemble(base)
all_models = {**base, **ensemb}

aapl_pipeline = train_pipeline(
    stock_data["AAPL"],
    "AAPL",
    models=all_models,
    verbose=False,
)

comparison_df = build_comparison_table(aapl_pipeline["results"])
print(comparison_df.to_string())

### 3.1 · Model Comparison Charts

In [ ]:
fig = plot_model_comparison(comparison_df)
plt.show()

### 3.2 · Best Model — Predictions and Residuals

In [ ]:
best_name = comparison_df.iloc[0]["Model"]
y_test    = aapl_pipeline["y_test"]
y_pred    = aapl_pipeline["predictions"][best_name]

print(f"Best model: {best_name}")

fig1 = plot_predictions(y_test, y_pred, best_name, ticker="AAPL")
plt.show()

fig2 = plot_residuals(y_test, y_pred, best_name)
plt.show()

### 3.3 · Feature Importance (Lasso Coefficients)

In [ ]:
lasso_model   = aapl_pipeline["results"]["Lasso Regression"]
lasso_trained = base["Lasso Regression"]   # fitted estimator lives in the dict
feat_cols_aapl = aapl_pipeline["feature_cols"]

# Re-retrieve the fitted model from the pipeline dict
from stock_prediction.models.train import build_models as _bm
_tmp = _bm()
_tmp["Lasso Regression"].fit(aapl_pipeline["X_train"], aapl_pipeline["y_train"])
fitted_lasso = _tmp["Lasso Regression"]

fig = plot_feature_importance(fitted_lasso, feat_cols_aapl,
                              title="AAPL: Lasso |Coefficients| — Top 20 Features")
plt.show()

# Zero-coefficient features (removed by Lasso)
n_zero = int(np.sum(fitted_lasso.coef_ == 0))
print(f"\nFeatures zeroed out by Lasso: {n_zero} / {len(feat_cols_aapl)} "
      f"({n_zero / len(feat_cols_aapl):.1%})")

## 4 · Cross-Stock Analysis

Run the best model (Lasso) across all 5 tickers.

In [ ]:
multi_results = run_all_stocks(stock_data, verbose=False)

# Build a summary table
rows = []
for ticker, out in multi_results.items():
    m = list(out["results"].values())[0]
    rows.append({
        "Ticker":   ticker,
        "Sector":   STOCKS[ticker],
        "R²":       round(m.r2,   4),
        "RMSE ($)": round(m.rmse, 2),
        "MAE ($)":  round(m.mae,  2),
        "Dir. Acc.": f"{m.directional_accuracy:.2%}",
    })

summary = pd.DataFrame(rows).sort_values("R²", ascending=False).reset_index(drop=True)
print(summary.to_string(index=False))
print(f"\nAverage R²: {summary['R²'].mean():.4f}")

### 4.1 · Cross-Stock Chart

In [ ]:
stock_metric_map = {
    ticker: {
        "r2":   list(out["results"].values())[0].r2,
        "rmse": list(out["results"].values())[0].rmse,
    }
    for ticker, out in multi_results.items()
}

fig = plot_cross_stock_comparison(stock_metric_map, sector_map=STOCKS)
plt.show()

## 5 · Key Findings

| Finding | Detail |
|---------|--------|
| **Best model** | Lasso Regression (α=0.01) — R²≈0.91 average |
| **Feature selection** | Lasso zeroes ~40% of features automatically |
| **Top feature** | `Price_Lag_5` — short-term autocorrelation dominates |
| **Most predictable sector** | Energy (XOM R²=0.977) |
| **Hardest sector** | Healthcare (JNJ R²=0.774) |
| **Directional accuracy** | ~52–54% — slightly better than random |
| **Tree models** | Negative R² — need return-relative normalisation |

See `README.md` for the full results tables and methodology.